# Round 5.4: A8 harder rule documents
**Runtime:** Colab T4. **Needs:** `Thesis-Coding-round5.4.zip` in `MyDrive/Thesis/` and the same `llm_cache.jsonl`.

Review `data/rule_docs_hard/` and `data/A8_HARD_DOCS_NOTES.md` BEFORE running: they are drafts. Results go to `results_a8` and never touch earlier results.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch, os
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

ZIP_PATH = "/content/drive/MyDrive/Thesis/Thesis-Coding-round5.4.zip"
A8 = "/content/drive/MyDrive/Thesis/results_a8"
os.makedirs(A8, exist_ok=True)
os.environ["USE_STUB_LLM"] = "false"
os.environ["LLM_MODEL_NAME"] = "Qwen/Qwen2.5-7B-Instruct"
os.environ["LLM_CACHE_PATH"] = "/content/drive/MyDrive/Thesis/llm_cache.jsonl"
os.environ["LLM_CACHE"] = "1"
os.environ["LLM_BATCH_SIZE"] = "8"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
!rm -rf /content/Thesis-Coding && mkdir -p /content/Thesis-Coding
!unzip -q -o "$ZIP_PATH" -d /content/Thesis-Coding
%cd /content/Thesis-Coding
assert "round 5.4" in open("CHANGES.md").read().lower(), "Old zip: upload Thesis-Coding-round5.4.zip"
%pip install -q -r requirements.txt "requests==2.32.4"
!pip freeze > $A8/pip_freeze.txt
!python -m src.pipeline
!python -m evaluation.verify_setup
!python -m pytest -q tests 2>&1 | tail -3

## Build a separate index from the harder documents

In [ ]:
!python -m src.pipeline --rule-docs-dir data/rule_docs_hard --faiss-index-path artifacts_hard/faiss.index --chunks-parquet-path artifacts_hard/chunks.parquet
HARD = "RULE_DOCS_DIR=data/rule_docs_hard ARTIFACTS_DIR=artifacts_hard FAISS_INDEX_PATH=artifacts_hard/faiss.index CHUNKS_PARQUET_PATH=artifacts_hard/chunks.parquet"

## A8 on the holdout split: LLM decides vs extract + compare vs ungrounded, on the HARD documents
Extraction prompts are new (8 cards), so a few generations happen; LLM-decides prompts are new too (about 40 minutes).

In [ ]:
!env $HARD python -m evaluation.run_ablation --split holdout --no-core --out-dir $A8/holdout_hard --arms A7 2>&1 | tr "\r" "\n" | grep --line-buffered -E "===|llm_decides|extract|Error|Traceback" | tee $A8/holdout_hard.log
!python -m evaluation.make_report --results-dir $A8/holdout_hard

## Send back
Zip `MyDrive/Thesis/results_a8/` (it includes `pip_freeze.txt` for your Methodology chapter) and push it to the repo.